<a href="https://colab.research.google.com/github/sgisgeodata/sgis-data-manual/blob/main/%EA%B0%9C%EB%B3%84%20%EA%B5%90%EC%9C%A1%EC%9E%90%EB%A3%8C%20Training%20Materials/(260716)%20%EA%B4%91%EC%A3%BC%EC%97%B0%EA%B5%AC%EC%9B%90/GeoAI%20%EC%8B%A4%EC%8A%B5/GeoDeep_building/geodeep_gwangju.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GeoDeep을 활용한 정사영상 객체 탐지 실습

이번 실습에서는 정사영상(GeoTIFF)에 사전학습된 AI 모델을 적용하여 객체를 탐지하고, 결과를 지도 위에 시각화한다.



## 0. GeoDeep과 GeoAI에 관한 간단한 설명

### 0.1 GeoDeep
[GeoDeep GitHub](https://github.com/uav4geo/GeoDeep)

**GeoDeep**은 GeoTIFF와 같은 지리공간 래스터 데이터를 대상으로 AI 기반 객체탐지를 수행하는 오픈소스 Python 라이브러리이다. 사전학습된 모델이 포함되어 있어 별도의 모델 학습 없이도 정사영상에서 차량, 도로, 건물 등 특정 객체를 탐지할 수 있다.

- **객체탐지(Object Detection)**: `detect()` 함수를 사용하며, 차량이나 항공영상 내 객체를 bounding box 형태로 탐지한다.
- **의미론적 분할(Semantic Segmentation)**: `segment()` 함수를 사용하며, 건물 또는 도로처럼 일정 영역을 픽셀 단위 마스크로 분할한다.

### 0.2 GeoAI

[GeoAI GitHub](https://github.com/opengeos/geoai)

**GeoAI**는 위성영상, 항공사진, 벡터 데이터 등을 대상으로 AI 모델 학습, 추론, 데이터 전처리, 시각화 등을 지원한다.

이번 실습에서는 주로 다음 용도로 사용한다.

- `geoai.raster_to_vector()`: 마스크 TIFF를 GeoJSON 폴리곤으로 변환
- `geoai.view_vector_interactive()`: 정사영상 위에 벡터 결과 시각화

### 0.3 실습 흐름

1. 실습 환경 설치 및 라이브러리 불러오기  
2. GitHub에 있는 GeoTIFF 정사영상 불러오기  
3. `segment()`를 이용한 건물 마스크 생성  
4. `detect()`를 이용한 객체 탐지
5. GeoAI 라이브러리를 이용하여 지도 위에 결과 시각화


## 1. 실습 환경 설치


In [ ]:
# GeoDeep, GeoAI 실습 패키지 설치
!pip install -q -U geodeep rasterio matplotlib geopandas shapely leafmap
!pip install -q -U geoai-py localtileserver


## 2. 라이브러리 불러오기
실습에 사용할 주요 라이브러리와 함수를 불러온다.

In [ ]:
# 기본 라이브러리
import geoai
import geopandas as gpd
import rasterio
import leafmap
from google.colab import files

# GeoDeep 모델 실행 함수
from geodeep import segment, detect

# GeoDeep 마스크 저장 함수
from geodeep.segmentation import save_mask_to_raster


## 3. 실습용 정사영상 불러오기

GitHub에 저장된 실습용 GeoTIFF를 불러온다.  

해당 자료는 **광주 첨단지구** 일대 정사영상을 실습용으로 클립한 GeoTIFF 파일입니다.

In [ ]:
# 실습용 정사영상 GitHub Raw URL
raster_url = "https://raw.githubusercontent.com/sgisgeodata/sgis-data-manual/main/개별 교육자료 Training Materials/(260716) 광주연구원/GeoAI 실습/GeoDeep_building/buildings.tif"

# 정사영상 다운로드
raster_path = geoai.download_file(raster_url)

In [ ]:
# 정사영상 확인 (이미지가 깨지면 재실행 해보기)
geoai.view_raster(raster_url)

## 4. GeoDeep `segment()`로 건물 마스크 생성

`segment()`는 정사영상에서 특정 객체 영역을 픽셀 단위로 분할합니다.


**기본 세그멘테이션 모델**

| 모델명 | 결과 |
|---|---|
| `buildings` | 건물 마스크 |
| `roads` | 도로 마스크 |


### 4.1. 모델 실행

여기서는 `buildings` 모델을 사용해 건물 후보 영역을 마스크로 생성합니다.

In [ ]:
# GeoDeep segmentation 모델 실행
mask = segment(raster_path, 'buildings')

### 4.2. 생성된 마스크를 GeoTIFF로 저장

생성된 마스크를 좌표 정보가 유지되는 GeoTIFF 파일로 저장한다.


In [ ]:
# 저장할 파일명 지정
building_mask = "building_mask.tif"
building_vector = "building.geojson"

# 마스크 배열을 원본 영상 좌표계가 유지된 GeoTIFF로 저장
save_mask_to_raster(
    raster_path,
    mask,
    building_mask
)


### 4.3. 마스크 TIFF를 GeoJSON으로 변환

마스크 래스터를 폴리곤 벡터로 변환하여 GeoJSON으로 저장한다.

In [ ]:
# 마스크 래스터를 벡터 폴리곤으로 변환
gdf = geoai.raster_to_vector(
    building_mask,
    building_vector
)

In [ ]:
# 건물 gdf 출력
gdf.head()

In [ ]:
# 폴리곤 개수 확인
print("건물 폴리곤 개수:", len(gdf))

### 4.4. 건물 마스크 결과를 지도 위에 시각화

`view_vector_interactive()`를 사용하면 원본 정사영상 위에 GeoJSON 벡터를 겹쳐 볼 수 있습니다.

- `gdf`: 방금 만든 건물 폴리곤
- `tiles=raster_url`: 배경으로 사용할 원본 정사영상


In [ ]:
# 원본 정사영상 위에 건물 마스크 폴리곤 시각화
geoai.view_vector_interactive(
    gdf,
    tiles=raster_url
)


## 5. GeoDeep `detect()`로 객체 탐지 실행

`detect()`는 객체별 위치를 **박스 또는 폴리곤 형태의 GeoJSON**으로 반환합니다.  
즉, `segment()`처럼 픽셀 마스크를 만드는 것이 아니라, 탐지된 객체의 경계와 속성값(class)을 저장합니다.

### 5.1. 모델 실행

실습에서는 여러 객체 클래스를 탐지할 수 있는 `waldo30_nano` 모델을 사용합니다.


In [ ]:
# GeoDeep object detection 모델 실행
waldo_geojson = detect(
    raster_path,
    "waldo30_nano",
    output_type="geojson"
)


### 5.2. 탐지 GeoJSON을 GeoDataFrame으로 읽기

저장한 GeoJSON을 `geopandas`로 읽어 `gdf_waldo`를 만듭니다.  

In [ ]:
# GeoJSON 파일을 GeoDataFrame으로 읽기
gdf_waldo = gpd.read_file(waldo_geojson)

In [ ]:
# gdf_waldo 출력
gdf_waldo.head()

In [ ]:
# 탐지 객체 수 확인
print("탐지 객체 수:", len(gdf_waldo))

In [ ]:
# 클래스별 탐지 개수 확인
if "class" in gdf_waldo.columns:
    print(gdf_waldo["class"].value_counts())

### 5.3. 객체 탐지 결과 시각화

탐지 결과에 `class` 컬럼이 있으면 객체 종류별로 색상을 구분해서 볼 수 있습니다.  

In [ ]:
# 원본 정사영상 위에 객체 탐지 결과 시각화
geoai.view_vector_interactive(
    gdf_waldo,
    column="class",
    tiles=raster_url
)

## 6. 결과 저장 후 QGIS에 불러오기
분석 결과를 QGIS에서 확인할 수 있도록 GeoPackage(`.gpkg`) 형식으로 저장합니다.  

In [ ]:
# QGIS용 GeoPackage 저장
gdf.to_file("building_result.gpkg", layer="building", driver="GPKG")
gdf_waldo.to_file("waldo_result.gpkg", layer="waldo", driver="GPKG")

In [ ]:
files.download("buildings.tif")
files.download("building_result.gpkg")
files.download("waldo_result.gpkg")

## 7. 모델을 바꿔가며 실습하기

GeoDeep에서는 크게 두 가지 방식의 모델을 사용합니다.

### 1) 마스크를 만드는 세그멘테이션 모델

```python
mask = segment(raster_path, "buildings")
mask = segment(raster_path, "roads")
```

### 2) 객체 탐지 하는 모델

```python
geojson = detect(raster_path, "cars", output_type="geojson")
geojson = detect(raster_path, "aerovision", output_type="geojson")
geojson = detect(raster_path, "waldo30_nano", output_type="geojson")
```


## 실습 내용 정리

1. GitHub의 GeoTIFF를 Colab에서 불러오기
2. GeoDeep `segment()`를 이용한 마스크 생성
3. GeoDeep `detect()`를 이용한 객체탐지
4. GeoAI를 이용한 마스크의 GeoJSON 변환
5. 원본 정사영상 위 결과 시각화

사전학습 모델은 영상 해상도와 촬영 지역에 따라 결과가 달라질 수 있으므로, 실습 결과는 모델 적용 가능성을 확인하는 예시로 해석합니다.
